# Verify the preserved P4b bounded-smoke artifact

Attach **exact Version 1** of the private dataset `thestonedape/task-aware-eeg2text-task-segmented-smoke`, enable Internet, and enable the private Kaggle secret `GITHUB_TOKEN`. Use a CPU session. This notebook independently re-hashes the sealed bounded-smoke output, never deserializes its checkpoints, and never accesses validation or held-out test data. A pass verifies preservation only; it is not a scientific arm comparison.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
VERIFIER_COMMIT = 'a9b214492a40c6ddb561ecd6cb279ff6cedd2a18'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-segmented-smoke-preserved-verification'
DATASET_SLUG = 'thestonedape/task-aware-eeg2text-task-segmented-smoke'
PRESERVED_DATASET_VERSION = 1
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eeg2text-task-segmented-smoke-version-1'
EXPECTED_MANIFEST_SHA256 = '2cf38c78bdc25815fbb16ad17832ac4062fe48dbc814bcf82c6999b2950ec2a3'
EXPECTED_METADATA_SHA256 = '91aa75ca77889627d4d2c42a104dd7063627d5b1996a67a81d65046d165b7f94'
EXPECTED_EXECUTION_CONTRACT_SHA256 = '9fd862b970ab95ded6f5efa0eb5290f8687bad7fd3a7f251f9ca66176de9f813'
EXPECTED_RUNNER_SOURCE_SHA256 = '62c246f6f8979b8a2f58b6b36072c7ac238d8c86a2fcdd0e3f2843af90cfef7c'
EXPECTED_REPORT_SHA256 = 'a2537e4a10de659705c956ec74105239e2046b5d42ca01a3524e1480b1081187'
EXPECTED_REPORT_BYTES = 2182
EXPECTED_ARMS = ['global_mixed', 'true_task_segmented', 'pseudo_task_segmented']
assert len(VERIFIER_COMMIT) == 40 and PRESERVED_DATASET_VERSION == 1
assert all(len(value) == 64 for value in (EXPECTED_MANIFEST_SHA256, EXPECTED_METADATA_SHA256, EXPECTED_EXECUTION_CONTRACT_SHA256, EXPECTED_RUNNER_SOURCE_SHA256, EXPECTED_REPORT_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    if os.path.exists(askpass):
        os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', VERIFIER_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == VERIFIER_COMMIT
subprocess.run([sys.executable, '-m', 'unittest', 'evaluation.test_verify_task_segmented_smoke_artifact'], check=True, cwd=WORKTREE)
print({'python': platform.python_version(), 'verifier_commit': actual_commit, 'regressions': 'PASS'})

In [ ]:
EXPECTED_TOP_LEVEL = {
    'arm_summary.csv', 'common_batch_trace.csv',
    'smoke_run_metadata.json', 'task_segmented_smoke_manifest.json', 'runs',
}
EXPECTED_ARM_DIRS = {'global_mixed', 'true_task_segmented', 'pseudo_task_segmented'}
EXPECTED_PER_ARM = {'resume_checkpoint.pt', 'run_summary.json', 'step_trace.csv'}
manifest_candidates = glob.glob('/kaggle/input/**/task_segmented_smoke_manifest.json', recursive=True)
artifact_roots = []
for path in manifest_candidates:
    root = os.path.dirname(path)
    try:
        names = set(os.listdir(root))
    except OSError:
        continue
    if names != EXPECTED_TOP_LEVEL:
        continue
    if os.path.islink(root) or any(os.path.islink(os.path.join(root, name)) for name in names):
        continue
    if not os.path.isdir(os.path.join(root, 'runs')):
        continue
    runs_root = os.path.join(root, 'runs')
    if set(os.listdir(runs_root)) != EXPECTED_ARM_DIRS:
        continue
    nested_ok = True
    for arm in EXPECTED_ARM_DIRS:
        arm_root = os.path.join(runs_root, arm)
        if os.path.islink(arm_root) or not os.path.isdir(arm_root) or set(os.listdir(arm_root)) != EXPECTED_PER_ARM:
            nested_ok = False
            break
        if any(os.path.islink(os.path.join(arm_root, name)) or not os.path.isfile(os.path.join(arm_root, name)) for name in EXPECTED_PER_ARM):
            nested_ok = False
            break
    if not nested_ok:
        continue
    artifact_roots.append(root)
artifact_roots = sorted(set(artifact_roots))
assert len(artifact_roots) == 1, ('Attach exact Version 1 of the one complete bounded-smoke dataset', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
manifest_path = os.path.join(ARTIFACT_ROOT, 'task_segmented_smoke_manifest.json')
assert digest(manifest_path) == EXPECTED_MANIFEST_SHA256
assert digest(os.path.join(ARTIFACT_ROOT, 'smoke_run_metadata.json')) == EXPECTED_METADATA_SHA256
print({'dataset_slug': DATASET_SLUG, 'dataset_version': PRESERVED_DATASET_VERSION, 'artifact_root': ARTIFACT_ROOT, 'manifest_sha256': EXPECTED_MANIFEST_SHA256})

In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
os.makedirs(OUTPUT)
report_path = os.path.join(OUTPUT, 'smoke_artifact_verification_report.json')
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'verify_task_segmented_smoke_artifact.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--expected-manifest-sha256', EXPECTED_MANIFEST_SHA256,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
    '--output-report', report_path,
], check=True, cwd=WORKTREE)
with open(report_path, encoding='utf-8') as handle:
    report = json.load(handle)
assert report['status'] == 'pass'
assert report['preserved_source_id'] == PRESERVED_SOURCE_ID
assert report['smoke_manifest_sha256'] == EXPECTED_MANIFEST_SHA256
assert report['execution_contract_sha256'] == EXPECTED_EXECUTION_CONTRACT_SHA256
assert report['project_commit'] == VERIFIER_COMMIT
assert report['runner_source_sha256'] == EXPECTED_RUNNER_SOURCE_SHA256
assert report['arms'] == EXPECTED_ARMS
assert report['optimizer_steps_per_arm'] == 2 and report['total_optimizer_steps'] == 6
assert report['common_trace_rows'] == 128 and report['checkpoint_deserialized'] is False
assert len(report['verified_artifact_sha256']) == 12
assert report['full_training_authorized'] is False
assert report['scientific_decision_permitted'] is False
assert report['held_out_test_accessed'] is False
report_sha256 = digest(report_path)
assert report_sha256 == EXPECTED_REPORT_SHA256
assert os.path.getsize(report_path) == EXPECTED_REPORT_BYTES
metadata = {
    'status': 'pass', 'schema_version': 1, 'verifier_commit': actual_commit,
    'preserved_source_id': PRESERVED_SOURCE_ID,
    'smoke_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'verification_report_sha256': EXPECTED_REPORT_SHA256,
    'verified_file_count': 13, 'checkpoint_deserialized': False,
    'full_training_authorized': False, 'scientific_decision_permitted': False,
    'held_out_test_accessed': False,
}
metadata_path = os.path.join(OUTPUT, 'verification_run_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
assert set(os.listdir(OUTPUT)) == {'smoke_artifact_verification_report.json', 'verification_run_metadata.json'}
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print({
    'status': report['status'], 'preserved_source_id': PRESERVED_SOURCE_ID,
    'smoke_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'verification_report_sha256': report_sha256,
    'verified_file_count': 13,
    'checkpoint_deserialized': report['checkpoint_deserialized'],
    'full_training_authorized': report['full_training_authorized'],
    'scientific_decision_permitted': report['scientific_decision_permitted'],
    'held_out_test_accessed': report['held_out_test_accessed'],
})
print('P4B BOUNDED SMOKE DATASET V1 CLEAN-REMOUNT VERIFICATION: PASS')

After the terminal PASS, send the printed verification-report SHA-256 back to the project vault. The smoke preservation gate is then closed, but the smoke still supplies no scientific arm comparison. The 45 fits remain unauthorized until the verified evidence is recorded and a separate full-run launch is approved.